In [1]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit import RDLogger
RDLogger.DisableLog('rdAll')

# --- Load & clean ---
with open("database.json") as f:
    raw = json.load(f)

df = pd.DataFrame.from_dict(raw, orient="index")
df.index.name = "cid"
df.replace("Not available", np.nan, inplace=True)

# Column name cleanup
df.rename(columns={
    "expt_s (cal/K.mol)": "expt_s",
    "d_expt_s (cal/K.mol)": "d_expt_s",
    "calc_s (cal/mol.K)": "calc_s",
    "d_calc_s (cal/mol.K)": "d_calc_s",
}, inplace=True)

# Fix dtypes
for col in ["expt_h", "d_expt_h", "expt_s", "d_expt_s"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Functional groups: list → sorted tuple
df["groups"] = df["groups"].apply(lambda x: tuple(sorted(x)))

# --- Molecular descriptors ---
mols = df["smiles"].apply(Chem.MolFromSmiles)

desc_funcs = {
    "MolWt": Descriptors.MolWt,
    "LogP": Descriptors.MolLogP,
    "HBA": Descriptors.NumHAcceptors,
    "HBD": Descriptors.NumHDonors,
    "TPSA": Descriptors.TPSA,
}

for name, func in desc_funcs.items():
    df[name] = mols.apply(func)

# --- Derived columns ---
df["residual"] = df["calc"] - df["expt"]
df["abs_residual"] = df["residual"].abs()

# Primary functional group label (first in sorted tuple, or "none")
df["primary_group"] = df["groups"].apply(lambda g: g[0] if g and g[0] != "" else "none")

# Exploded groups for per-group stats
df_groups = df[["residual", "abs_residual", "expt", "groups"]].explode("groups")
df_groups = df_groups[df_groups["groups"] != ""]

# Display name for hover
df["display_name"] = df["nickname"].str.strip().where(
    df["nickname"].str.strip().str.len() > 0, df["iupac"].str[:30]
)

print(f"Loaded {len(df)} molecules, {len(desc_funcs)} descriptors computed.")
print(f"MAE: {df['abs_residual'].mean():.2f} kcal/mol | Median AE: {df['abs_residual'].median():.2f} kcal/mol")

Loaded 642 molecules, 5 descriptors computed.
MAE: 1.11 kcal/mol | Median AE: 0.95 kcal/mol


## FreeSolv: Two Questions About Molecular Solvation

**What is this?** FreeSolv contains 642 small molecules with both experimental and simulated hydration free energies (ΔG, kcal/mol). Hydration free energy measures how favorably a molecule dissolves in water — more negative means more hydrophilic.

**Two questions:**
1. **What molecular properties predict how well a molecule dissolves in water?**
2. **Where do physics-based simulations fail — and is there a pattern?**

### How accurate is the simulation?

In [2]:
# --- Plot 1: Simulation vs Experiment (interactive, filterable by functional group) ---

# Get groups with enough molecules for meaningful filtering
group_counts = df_groups["groups"].value_counts()
filter_groups = sorted(group_counts[group_counts >= 10].index.tolist())

# Count bad predictions (|error| > 4) per group
bad_counts = (df_groups[df_groups["abs_residual"] > 4]
              .groupby("groups").size()
              .reindex(filter_groups, fill_value=0))

# MAE per group
group_mae = df_groups.groupby("groups")["abs_residual"].mean()

fig = go.Figure()

# "All" trace (shown by default)
fig.add_trace(go.Scatter(
    x=df["expt"], y=df["calc"],
    mode="markers",
    marker=dict(
        size=7,
        color=df["abs_residual"],
        colorscale="RdYlGn_r",
        cmin=0, cmax=6,
        colorbar=dict(title="|Error|<br>(kcal/mol)"),
        opacity=0.7,
        line=dict(width=0.3, color="grey"),
    ),
    customdata=np.column_stack([
        df["display_name"], df["residual"].round(2),
        df["abs_residual"].round(2), df["primary_group"],
        df["TPSA"].round(1), df["MolWt"].round(1),
    ]),
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Expt ΔG: %{x:.2f} kcal/mol<br>"
        "Calc ΔG: %{y:.2f} kcal/mol<br>"
        "Error: %{customdata[1]} kcal/mol<br>"
        "Group: %{customdata[3]}<br>"
        "TPSA: %{customdata[4]} | MW: %{customdata[5]}"
        "<extra></extra>"
    ),
    visible=True,
    name="All molecules",
))

# One trace per functional group (hidden by default)
for grp in filter_groups:
    mask = df_groups[df_groups["groups"] == grp].index
    dfg = df.loc[df.index.isin(mask)]
    fig.add_trace(go.Scatter(
        x=dfg["expt"], y=dfg["calc"],
        mode="markers",
        marker=dict(
            size=7,
            color=dfg["abs_residual"],
            colorscale="RdYlGn_r",
            cmin=0, cmax=6,
            colorbar=dict(title="|Error|<br>(kcal/mol)"),
            opacity=0.7,
            line=dict(width=0.3, color="grey"),
        ),
        customdata=np.column_stack([
            dfg["display_name"], dfg["residual"].round(2),
            dfg["abs_residual"].round(2), dfg["primary_group"],
            dfg["TPSA"].round(1), dfg["MolWt"].round(1),
        ]),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Expt ΔG: %{x:.2f} kcal/mol<br>"
            "Calc ΔG: %{y:.2f} kcal/mol<br>"
            "Error: %{customdata[1]} kcal/mol<br>"
            "Group: %{customdata[3]}<br>"
            "TPSA: %{customdata[4]} | MW: %{customdata[5]}"
            "<extra></extra>"
        ),
        visible=False,
        name=grp,
    ))

# Base annotation template
mae_all = df["abs_residual"].mean()
mae_annotation = dict(
    x=0.02, y=0.98, xref="paper", yref="paper",
    showarrow=False, font=dict(size=13),
    bgcolor="white", bordercolor="grey", borderwidth=1,
)
filter_label = dict(
    text="Filter by group:", showarrow=False,
    x=0.95, xanchor="right", y=1.16, yanchor="top",
    xref="paper", yref="paper", font=dict(size=11),
)

# Dropdown buttons — each updates visibility AND the MAE annotation
n_bad_all = (df["abs_residual"] > 4).sum()
buttons = [dict(
    label=f"All molecules (n={len(df)}, {n_bad_all} bad)",
    method="update",
    args=[
        {"visible": [True] + [False] * len(filter_groups)},
        {"annotations": [
            {**mae_annotation, "text": f"MAE = {mae_all:.2f} kcal/mol"},
            filter_label,
        ]},
    ],
)]
for i, grp in enumerate(filter_groups):
    vis = [False] * (1 + len(filter_groups))
    vis[i + 1] = True
    n = group_counts[grp]
    n_bad = bad_counts[grp]
    grp_mae = group_mae[grp]
    label = f"{grp} (n={n}, {n_bad} bad)" if n_bad > 0 else f"{grp} (n={n})"
    buttons.append(dict(
        label=label,
        method="update",
        args=[
            {"visible": vis},
            {"annotations": [
                {**mae_annotation, "text": f"MAE = {grp_mae:.2f} kcal/mol"},
                filter_label,
            ]},
        ],
    ))

# Reference line
line_min = min(df["expt"].min(), df["calc"].min()) - 1
line_max = max(df["expt"].max(), df["calc"].max()) + 1

fig.add_shape(type="line", x0=line_min, y0=line_min, x1=line_max, y1=line_max,
              line=dict(color="red", dash="dash", width=1.5))

fig.update_layout(
    title=dict(text="Calc vs Expt ΔG", x=0.02, xanchor="left"),
    xaxis_title="Experimental ΔG (kcal/mol)",
    yaxis_title="Calculated ΔG (kcal/mol)",
    margin=dict(t=100, r=100),
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        x=0.95, xanchor="right",
        y=1.12, yanchor="top",
        showactive=True,
        bgcolor="white",
    )],
    annotations=[
        {**mae_annotation, "text": f"MAE = {mae_all:.2f} kcal/mol"},
        filter_label,
    ],
    width=750, height=650,
    template="plotly_white",
)

fig.show()

The simulation reproduces experimental ΔG within ~1.1 kcal/mol on average — most points cluster tightly around the diagonal. But a handful of molecules are off by 5–10 kcal/mol. Use the dropdown to filter by functional group and see which chemical families the simulation handles well vs. poorly.

### Q1: What predicts solvation?

In [3]:
# --- Plot 2: What predicts solvation? (interactive descriptor toggle) ---

descriptors = {
    "TPSA": "Topological Polar Surface Area (Å²)",
    "LogP": "Partition Coefficient (LogP)",
    "HBD": "H-Bond Donors",
    "HBA": "H-Bond Acceptors",
    "MolWt": "Molecular Weight (Da)",
}

fig = go.Figure()

for desc_col in descriptors:
    r = df[desc_col].corr(df["expt"])
    fig.add_trace(go.Scatter(
        x=df[desc_col], y=df["expt"],
        mode="markers",
        marker=dict(size=6, opacity=0.5, color="steelblue",
                    line=dict(width=0.3, color="grey")),
        customdata=np.column_stack([
            df["display_name"],
            df[desc_col].round(2),
            df["expt"].round(2),
        ]),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            f"{desc_col}: " + "%{customdata[1]}<br>"
            "Expt ΔG: %{customdata[2]} kcal/mol"
            "<extra></extra>"
        ),
        visible=False,
        name=f"{desc_col} (r={r:.2f})",
    ))

# TPSA visible by default (first trace)
fig.data[0].visible = True

# Dropdown buttons
buttons = []
for i, (desc_col, desc_label) in enumerate(descriptors.items()):
    r = df[desc_col].corr(df["expt"])
    vis = [False] * len(descriptors)
    vis[i] = True
    buttons.append(dict(
        label=f"{desc_col} (r={r:.2f})",
        method="update",
        args=[
            {"visible": vis},
            {"xaxis.title.text": desc_label},
        ],
    ))

fig.update_layout(
    title="What Predicts Hydration Free Energy?",
    xaxis_title=list(descriptors.values())[0],
    yaxis_title="Experimental ΔG (kcal/mol)",
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        x=1.0, xanchor="right",
        y=1.15, yanchor="top",
        showactive=True,
        bgcolor="white",
    )],
    width=750, height=550,
    template="plotly_white",
)

fig.show()

TPSA (polar surface area) predicts solvation with r = −0.74 — toggle to LogP (r = 0.34) and watch the cloud lose its structure. The standard hydrophobicity metric is a weaker predictor than a simple measure of polarity. HBD (r = −0.69) and HBA (r = −0.60) confirm this — every measure of polar, hydrogen-bonding capacity outperforms the standard hydrophobicity metric. Solvation is driven by specific polar interactions (H-bonds, electrostatics), not bulk hydrophobicity.

### Q2: Where does the simulation fail?

In [4]:
# --- Plot 3: Median |error| by functional group + worst individual predictions ---

resid_stats = (df_groups.groupby("groups")["abs_residual"]
               .agg(["count", "median"])
               .query("count >= 10")
               .sort_values("median", ascending=True))

fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.55, 0.45],
    vertical_spacing=0.12,
    subplot_titles=[
        "Median Simulation Error by Functional Group (n ≥ 10)",
        "Worst Individual Predictions (|error| > 4 kcal/mol)",
    ],
)

# Top: horizontal bar chart
fig.add_trace(go.Bar(
    y=resid_stats.index,
    x=resid_stats["median"],
    orientation="h",
    marker=dict(
        color=resid_stats["median"],
        colorscale="OrRd",
        cmin=0, cmax=resid_stats["median"].max(),
    ),
    customdata=resid_stats["count"].values,
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Median |error|: %{x:.2f} kcal/mol<br>"
        "n = %{customdata}"
        "<extra></extra>"
    ),
    showlegend=False,
), row=1, col=1)

# Bottom: outlier lollipop chart
outliers = df[df["abs_residual"] > 4].copy()

# Assign chemical class
def classify(row):
    grps = " ".join(row["groups"]).lower() if row["groups"] else ""
    name = row["display_name"].lower()
    if any(x in name for x in ["glucose", "xylose", "mannitol"]):
        return "Sugar / sugar alcohol"
    if any(x in grps for x in ["thiophosphoric", "phosphoric"]):
        return "Organophosphate"
    return "Other"

outliers["chem_class"] = outliers.apply(classify, axis=1)
outliers = outliers.sort_values("residual")

class_colors = {
    "Sugar / sugar alcohol": "#2ca02c",
    "Organophosphate": "#d62728",
    "Other": "#7f7f7f",
}

for cls in class_colors:
    sub = outliers[outliers["chem_class"] == cls]
    if len(sub) == 0:
        continue
    fig.add_trace(go.Scatter(
        x=sub["residual"],
        y=sub["display_name"],
        mode="markers",
        marker=dict(size=10, color=class_colors[cls]),
        name=cls,
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Error (calc − expt): %{x:.2f} kcal/mol<br>"
            f"Class: {cls}"
            "<extra></extra>"
        ),
        showlegend=True,
    ), row=2, col=1)

fig.add_vline(x=0, line_dash="dash", line_color="grey", row=2, col=1)

# Force all y-tick labels to show
fig.update_yaxes(dtick=1, row=1, col=1)
fig.update_yaxes(dtick=1, row=2, col=1)

fig.update_layout(
    height=900, width=800,
    margin=dict(t=40),
    template="plotly_white",
    legend=dict(x=0.7, y=0.35),
)
fig.update_xaxes(title_text="Median |Error| (kcal/mol)", row=1, col=1)
fig.update_xaxes(title_text="Error: calc − expt (kcal/mol)", row=2, col=1)

fig.show()

Thiophosphoric acid esters (median |error| = 3.7 kcal/mol) and carboxylic acids (2.7) are the hardest functional groups for the simulation. Aldehydes (0.4) and ketones (0.4) are easiest.

The 16 worst individual predictions (|error| > 4 kcal/mol) include recognizable clusters — sugars and sugar alcohols (glucose, xylose, mannitol) and organophosphates — but the rest are chemically diverse (caffeine, ketoprofen, endosulfan, halogenated biphenyls). These molecules also tend to have the most extreme ΔG values in the dataset — larger absolute errors may partly reflect scale (bigger number, bigger absolute error), not purely chemical difficulty.

In [5]:
# --- Plot 4: Is the error driven by scale? ---

from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression

# Scale correlation
r_scale, _ = pearsonr(df["expt"].abs(), df["abs_residual"])

# Partial correlation: TPSA vs |error| after removing scale effect
r_tpsa_raw, _ = pearsonr(df["TPSA"], df["abs_residual"])
lr = LinearRegression().fit(df[["expt"]].abs(), df["abs_residual"])
error_resid = df["abs_residual"] - lr.predict(df[["expt"]].abs())
r_tpsa_partial, _ = pearsonr(df["TPSA"], error_resid)

print(f"|expt| vs |error|:  r = {r_scale:.2f}")
print(f"TPSA vs |error| (raw):            r = {r_tpsa_raw:.2f}")
print(f"TPSA vs |error| (scale removed):  r = {r_tpsa_partial:.2f}")

# Plot
outlier_mask = df["abs_residual"] > 4
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.loc[~outlier_mask, "expt"].abs(),
    y=df.loc[~outlier_mask, "abs_residual"],
    mode="markers",
    marker=dict(size=6, opacity=0.4, color="steelblue",
                line=dict(width=0.3, color="grey")),
    customdata=np.column_stack([
        df.loc[~outlier_mask, "display_name"],
        df.loc[~outlier_mask, "expt"].round(2),
        df.loc[~outlier_mask, "abs_residual"].round(2),
    ]),
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "|Expt ΔG|: %{x:.2f} kcal/mol<br>"
        "|Error|: %{customdata[2]} kcal/mol"
        "<extra></extra>"
    ),
    name="All molecules",
))

fig.add_trace(go.Scatter(
    x=df.loc[outlier_mask, "expt"].abs(),
    y=df.loc[outlier_mask, "abs_residual"],
    mode="markers+text",
    marker=dict(size=9, color="crimson", opacity=0.8,
                line=dict(width=0.5, color="darkred")),
    text=df.loc[outlier_mask, "display_name"],
    textposition="top right",
    textfont=dict(size=8),
    customdata=np.column_stack([
        df.loc[outlier_mask, "display_name"],
        df.loc[outlier_mask, "expt"].round(2),
        df.loc[outlier_mask, "abs_residual"].round(2),
    ]),
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "|Expt ΔG|: %{x:.2f} kcal/mol<br>"
        "|Error|: %{customdata[2]} kcal/mol"
        "<extra></extra>"
    ),
    name="Outliers (|error| > 4)",
))

fig.add_annotation(
    x=0.98, y=0.02, xref="paper", yref="paper",
    xanchor="right", yanchor="bottom",
    text=f"r = {r_scale:.2f}",
    showarrow=False, font=dict(size=13),
    bgcolor="white", bordercolor="grey", borderwidth=1,
)

fig.update_layout(
    xaxis_title="|Experimental ΔG| (kcal/mol)",
    yaxis_title="|Simulation Error| (kcal/mol)",
    width=750, height=500,
    template="plotly_white",
    legend=dict(x=0.5, xanchor="center", y=1.05, yanchor="bottom",
                orientation="h"),
)

fig.show()

|expt| vs |error|:  r = 0.40
TPSA vs |error| (raw):            r = 0.36
TPSA vs |error| (scale removed):  r = 0.08


The outliers from Plot 3 aren't special — they sit at the extreme end of a general trend: molecules with larger |ΔG| have larger absolute errors (r ≈ 0.40). After removing this scale effect, TPSA vs |error| drops from r = 0.36 to r = 0.08 — the apparent link between polarity and simulation error was an artifact of magnitude, not chemistry.

---

## Summary

**Q1 — What predicts solvation?** Polar surface area (TPSA, r = −0.74) is the strongest single predictor of hydration free energy — stronger than LogP (r = 0.34), the standard hydrophobicity metric. HBD (r = −0.69) and HBA (r = −0.60) reinforce this. Solvation is driven by specific polar interactions, not bulk hydrophobicity.

**Q2 — Where do simulations fail?** The simulation reproduces experimental ΔG well on average (MAE ≈ 1.1 kcal/mol), but the 16 worst predictions include sugars/sugar alcohols and organophosphates alongside chemically diverse outliers. After controlling for ΔG magnitude, descriptor-error correlations vanish (TPSA vs |error|: r = 0.36 → 0.08) — the failures track with extreme values, not specific molecular properties.

**Dashboard direction:** Interactive exploration of simulation accuracy — filtering by molecular properties, functional groups, and error magnitude — to let users investigate which molecules and chemical families the simulation handles well or poorly.